#### 01. CARREGANDO MAPEAMENTO DE PASTAS E IMPORTS

In [ ]:
# Importando bibliotecas
from functions import *
import pandas as pd
import locale
from pathlib import Path
import shutil
from datetime import datetime
import re
import shutil


timer = Temporizador()

timer.iniciar()

locale.setlocale(locale.LC_TIME, 'Portuguese_Brazil.1252')  # Para Windows
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.options.display.float_format = lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

# Detecta se o script está sendo executado de um .py ou de um notebook
try:
    caminho_base = Path(__file__).resolve().parent
except NameError:
    # __file__ não existe em Jupyter ou ambiente interativo
    caminho_base = Path.cwd()

pasta_input_orcamento = caminho_base.parent / '01_INPUT_PIPELINE/03_ORCAMENTO_ANUAL'
pasta_staging_parquet = caminho_base.parent / '02_STAGING_PARQUET'

print("✅ Mapeamento de pastas concluído com sucesso!")

✅ Mapeamento de pastas concluído com sucesso!


#### 02. CARREGANDO ORCAMENTO ANUAL

In [43]:
# Carregar DIM_PRODUTOS_KRONA uma única vez
df_dim_produtos_krona = pd.read_parquet(pasta_staging_parquet / 'DIM_PRODUTOS_KRONA.parquet')

# Lista para armazenar os orçamentos processados
lista_orcamentos = []

# Percorrer todos os arquivos da pasta de orçamento
for origem in pasta_input_orcamento.iterdir():

	# Processar somente arquivos que contenham ORCAMENTO_KRONA no nome
	if origem.is_file() and "ORCAMENTO_KRONA" in origem.stem.upper():

		# Captar o ano no final do nome do arquivo
		match_ano = re.search(r'(\d{4})$', origem.stem)

		# Ignorar arquivo caso não tenha ano válido no final do nome
		if not match_ano:
			print(f"Arquivo ignorado - ano não identificado: {origem.name}")
			continue

		ano_orcamento = int(match_ano.group(1))

		# Definir arquivo temporário mantendo a extensão original
		copia = pasta_input_orcamento / f"{origem.stem}_TEMP{origem.suffix}"

		# Excluir arquivo temporário anterior, caso exista
		if copia.exists():
			copia.unlink()

		# Criar uma cópia temporária do arquivo original
		shutil.copy2(origem, copia)

		print(f"Processando: {origem.name} | Ano: {ano_orcamento}")

		# Carregar o arquivo temporário
		df_orcamento_excel = pd.read_excel(copia, sheet_name='ORCAMENTO', engine='calamine')

		# Excluir arquivo temporário
		if copia.exists():
			copia.unlink()

		# -----------------------------------------------------------------------
		# Tratar os dados do dataframe carregado
		# -----------------------------------------------------------------------

		# Garantir Cod_Produto como texto e completar com zeros à esquerda até 4 dígitos
		df_orcamento_excel["Cod_Produto"] = pd.to_numeric(df_orcamento_excel["Cod_Produto"], errors="coerce").astype("Int64").astype("string").str.zfill(4)

		# Colunas dimensionais
		colunas_dimensao = ["Marca", "Grupo", "Familia", "REGIONAL Ajustada", "Segmento Ajustado", "Cod_Produto", "Des_Produto_Com_Codigo"]

		# Colunas mensais de KG e VALOR
		colunas_kg = list(range(1, 13))
		colunas_valor = [f"{i}.1" for i in range(1, 13)]

		# Transformar KG para formato long
		df_kg = df_orcamento_excel.melt(id_vars=colunas_dimensao, value_vars=colunas_kg, var_name="MES", value_name="ORC_KG")

		# Transformar VALOR para formato long
		df_valor = df_orcamento_excel.melt(id_vars=colunas_dimensao, value_vars=colunas_valor, var_name="MES", value_name="ORC_VAL")

		# Ajustar o mês do bloco de VALOR para inteiro
		df_valor["MES"] = df_valor["MES"].str.replace(".1", "", regex=False).astype(int)

		# Juntar KG e VALOR pelas dimensões e pelo mês
		df_orcamento_excel = df_kg.merge(df_valor, on=colunas_dimensao + ["MES"], how="left")

		# Criar PERIODO usando o ano identificado no nome do arquivo
		df_orcamento_excel["PERIODO"] = pd.to_datetime(dict(year=ano_orcamento, month=df_orcamento_excel["MES"], day=1))

		# Excluir colunas auxiliares
		df_orcamento_excel = df_orcamento_excel.drop(columns=["MES", "Des_Produto_Com_Codigo"])

		# Renomear colunas
		df_orcamento_excel = df_orcamento_excel.rename(columns={
			"Marca": "MARCA",
			"Grupo": "GRUPO",
			"Familia": "FAMILIA",
			"REGIONAL Ajustada": "REGIONAL",
			"Segmento Ajustado": "SEGMENTO",
			"Cod_Produto": "COD_PROD"
		})

		# Mesclar com a dimensão de produtos para obter a descrição oficial
		df_orcamento_excel = df_orcamento_excel.merge(df_dim_produtos_krona[["COD_PROD", "DESC_PROD"]], on="COD_PROD", how="left")

		# Organizar o layout final das colunas
		colunas_final = ["MARCA", "GRUPO", "REGIONAL", "SEGMENTO", "COD_PROD", "DESC_PROD", "FAMILIA", "PERIODO", "ORC_KG", "ORC_VAL"]
		df_orcamento_excel = df_orcamento_excel[colunas_final]

		# Adicionar orçamento processado à lista
		lista_orcamentos.append(df_orcamento_excel)

# Consolidar todos os anos processados
BD_HISTORICO_ORCAMENTO = pd.concat(lista_orcamentos, ignore_index=True)

# Salvar histórico consolidado
BD_HISTORICO_ORCAMENTO.to_parquet(pasta_staging_parquet / "BD_HISTORICO_ORCAMENTO.parquet", index=False)

Processando: ORCAMENTO_KRONA_2026.xlsb | Ano: 2026


In [23]:
print(df_orcamento_excel.columns.tolist())

['Marca', 'Grupo', 'Familia', 'REGIONAL Ajustada', 'Segmento Ajustado', 'Cod_Produto', 'Des_Produto_Com_Codigo', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 'x', '1.1', '2.1', '3.1', '4.1', '5.1', '6.1', '7.1', '8.1', '9.1', '10.1', '11.1', '12.1']


In [ ]:
timer.finalizar()
print("🎯 Processo concluído com sucesso!")